<a href="https://colab.research.google.com/github/gregoirehendrix/Master_Thesis/blob/main/drainage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import math

G = 9.81   # gravitational acceleration [m/s^2]

def rho_solar_salt(T_K):
    T_C = T_K - 273.15
    return 2090.0 - 0.636 * T_C


def mu_solar_salt(T_K):
    T_C = T_K - 273.15
    return 1e-3 * (
        22.714
        - 0.120    * T_C
        + 2.281e-4 * T_C**2
        - 1.474e-7 * T_C**3
    )


def circular_pipe_geometry(D, fill_ratio):
    h     = fill_ratio * D
    r     = D / 2.0
    theta = math.acos((r - h) / r)   # half-angle subtended by free surface [rad]
    A     = r**2 * (theta - math.sin(theta) * math.cos(theta))
    P_w   = 2.0 * r * theta
    R_h   = A / P_w
    T     = 2.0 * r * math.sin(theta)
    D_h   = 4.0 * R_h
    return A, P_w, R_h, T, D_h


def colebrook_white(Re_h, D_h, eps, max_iter=200, tol=1e-10):
    rel_rough = eps / D_h
    # Swamee-Jain initial guess
    f = 0.25 / (math.log10(rel_rough / 3.7 + 5.74 / Re_h**0.9))**2
    for _ in range(max_iter):
        rhs   = -2.0 * math.log10(rel_rough / 3.7 + 2.51 / (Re_h * math.sqrt(f)))
        f_new = 1.0 / rhs**2
        if abs(f_new - f) < tol:
            return f_new
        f = f_new
    return f


def drainage_time(D_int, slope, L, T_C, fill_ratio, eps=46e-6, verbose=True):
    T_K = T_C + 273.15

    # --- Fluid properties ---
    rho = rho_solar_salt(T_K)
    mu  = mu_solar_salt(T_K)

    # --- Section geometry ---
    A, P_w, R_h, T_surf, D_h = circular_pipe_geometry(D_int, fill_ratio)
    h = fill_ratio * D_int

    # --- Coupled iteration: v <-> f_D via Re_h ---
    # Initial guess: f_D = 0.02 (typical turbulent)
    f_D = 0.02
    for _ in range(300):
        v     = math.sqrt(8.0 * G * R_h * slope / f_D)
        Re_h  = rho * v * D_h / mu
        if Re_h < 2300:
            f_D_new = 64.0 / Re_h
        else:
            f_D_new = colebrook_white(Re_h, D_h, eps)
        if abs(f_D_new - f_D) < 1e-10:
            f_D = f_D_new
            break
        f_D = f_D_new

    # Final consistent values
    v    = math.sqrt(8.0 * G * R_h * slope / f_D)
    Re_h = rho * v * D_h / mu
    Fr   = v / math.sqrt(G * h)

    if Re_h < 2300:
        regime = "laminar"
    elif Re_h < 4000:
        regime = "transition"
    else:
        regime = "turbulent"

    # --- Flow rate and drainage time ---
    Q     = v * A
    V     = (math.pi * D_int**2 / 4.0) * L
    t_s   = V / Q
    t_min = t_s / 60.0

    if verbose:
        print(f"Drainage = {t_s:.0f} s = {t_min:.1f} min")

    return {
        "rho": rho, "mu": mu, "A": A, "R_h": R_h, "D_h": D_h,
        "v": v, "f_D": f_D, "Re_h": Re_h, "Fr": Fr,
        "Q_m3s": Q, "V_m3": V,
        "t_s": t_s, "t_min": t_min,
    }



if __name__ == "__main__":

    FILL = 0.53    # fill ratio h/D
    EPS  = 46e-6   # pipe roughness [m]

    # --- User inputs ---
    DN    = float(input("DN [mm]            = "))
    slope = float(input("Slope [%]          = "))
    L     = float(input("Length [m]         = "))
    T     = float(input("Temperature [°C]   = "))

    #print("DN =", DN," mm, L=", L,"m, slope=",slope,"%, T=",T,"C")

    drainage_time(
        D_int=DN / 1000.0,
        slope=slope / 100.0,
        L=L,
        T_C=T,
        fill_ratio=FILL,
        eps=EPS
    )

DN [mm]            = 50
Slope [%]          = 0.5
Length [m]         = 600
Temperature [°C]   = 450
Drainage = 2522 s = 42.0 min
